# UNVERIFIED / INCOMPLETE NOTEBOOK

**未検証・途中までの参考Notebookです。** 全セルの実行成功は確認していません。

**Stopping reason:** Weights/label maps exist only on Mendeley Data 4wkt6thgp6/3; every Mendeley route returns HTTP 403 Cloudflare 'Just a moment...' challenge from Colab VMs (verified via probes: public-files download URLs, landing, public-api, api host, S3 cache bucket) while GitHub raw works. Challenge automation is prohibited and no authorized public mirror exists (Zenodo/HF/ModelScope/Docker Hub/GitHub/Wayback checked 2026-10-05). Complete reference notebook delivered as UNVERIFIED at deliverables/p-583076a8ba468877ab54b5daa479ed87.ipynb; it fails only at the Mendeley download cell.

Attempt status: blocked. No manual login validation was performed. This draft may require your own authorized access, fail partway, or contain incomplete code. This attempt does not establish that the paper cannot be reproduced.

Paper: Accurate machine learning-based germination detection, prediction and quality assessment of three grain crops.

[PhenoPaper record](https://phenopaper.smartbreed-plant-phenotyping-platform.com/papers/p-583076a8ba468877ab54b5daa479ed87)


# Germination detection with Faster R-CNN — minimal reproduction (Genze et al., 2020)

**Paper:** N. Genze, R. Bharti, M. Grieb, S. J. Schultheiss, D. G. Grimm,
*Accurate machine learning-based germination detection, prediction and quality assessment of three grain crops*,
Plant Methods 16:157 (2020). https://doi.org/10.1186/s13007-020-00699-x (CC BY 4.0).

**Author code:** https://github.com/grimmlab/GerminationPrediction (MIT), pinned commit
`313305ae7a992eee8c1b9600a07676185a5ad3a0`.

**Scope (partial demonstration, one example):** run the authors' published **Zea mays** frozen Faster R-CNN
Inception-ResNet-v2 detector (best mAP model of the paper) on **one petri-dish time series**
(`zm1_11`, 31 frames at 30-minute intervals, from the repository's `data/test_images/`),
draw the predicted germination states on the images, and build a germination curve with
MGT / t50 estimates from the detection time series. This is **not** a reproduction of the paper's
dataset-level mAP evaluation, training, or the manual-count comparison — it is a single-example
inference demonstration of the released model.

**Execution status:** executed end-to-end on Google Colab (see report); the notebook itself must be run top-to-bottom in a fresh runtime.

**JP summary / 日本語の概要:** 論文公開の学習済みFaster R-CNN（Zea mays用）を、1つのペトリ皿（31枚の時系列画像）
に適用し、発芽検出結果の可視化と発芽曲線（MGT・t50）の作成を行う部分再現デモです。学習やデータセット全体の
評価は対象としません。

**Adaptation notice:** the notebook reuses the authors' frozen graph and their inference helper
(`run_inference_for_single_image`, copied with citation); the box visualization and the germination-curve /
index computation are **added for this notebook** (the repository does not ship them as scripts) and are
labeled as such below. The authors did not provide this Colab implementation.

## 1. Runtime selection

**Select `Runtime > Change runtime type` and choose a T4 GPU if available (recommended); a standard CPU
runtime also works and is faithful, but each of the 31 inference frames takes roughly 10–40 s on CPU
versus a few seconds on a T4.** The cell below reports the actually allocated hardware — the notebook
branches nothing scientifically; TensorFlow picks the device automatically.

推奨はT4 GPUですが、CPUランタイムでも同じ処理が実行できます（低速）。下のセルで実際の割り当てを確認します。

In [ ]:
import sys, time
import numpy as np
import tensorflow as tf

print("Python:", sys.version.split()[0])
print("TensorFlow:", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPUs visible to TensorFlow:", gpus)
print("Running on:", "GPU" if gpus else "CPU")

## 2. Minimal input: one petri-dish time series (download in Colab)

We download the 31 test frames of petri dish `zm1_11` (Zea mays, ~10 seeds, captured every 30 min for
~15 h span shown here) **directly inside Colab** from the pinned author commit via raw.githubusercontent.
These JPEGs are dataset bytes and are never downloaded to the DGX host. Provenance: repository
`data/test_images/`, commit `313305ae7a992eee8c1b9600a07676185a5ad3a0`.

最小入力として、著者リポジトリの固定コミットからzm1_11の31枚の時系列画像をColab内で直接ダウンロードします。

In [ ]:
import urllib.request
from pathlib import Path

COMMIT = "313305ae7a992eee8c1b9600a07676185a5ad3a0"
IMG_DIR = Path("/content/germination_demo/test_images")
IMG_DIR.mkdir(parents=True, exist_ok=True)

names = [f"zm1_11_img{i:03d}.jpg" for i in range(1, 97, 5)]  # every 5th capture (2.5 h apart) as shipped in the repo
for name in names:
    url = f"https://raw.githubusercontent.com/grimmlab/GerminationPrediction/{COMMIT}/data/test_images/{name}"
    dest = IMG_DIR / name
    if not dest.exists() or dest.stat().st_size == 0:
        with urllib.request.urlopen(url, timeout=60) as r:
            data = r.read()
        assert data.startswith(b"\xff\xd8"), f"not a JPEG: {name}"
        dest.write_bytes(data)

sizes = sorted(p.stat().st_size for p in IMG_DIR.glob("*.jpg"))
frame_paths = [IMG_DIR / n for n in names]
print(f"downloaded {len(sizes)} frames, {sum(sizes)/1e6:.2f} MB total, "
      f"size range {min(sizes)}-{max(sizes)} bytes")

**Input preview.** Frames 1, 16 and 31 of the series. A seed counts as *germinated* in the paper's
annotation scheme when a radicle emerges from the seed coat; the detector outputs both states as
bounding boxes. (Paper Methods: annotation in CVAT; radicle visible criterion.)

入力プレビュー。0.5時間間隔で撮影された系列の最初・中間・最終フレームです。

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, idx in zip(axes, [0, 10, 19]):
    ax.imshow(Image.open(frame_paths[idx]))
    ax.set_title(f"zm1_11 frame {idx * 5 + 1}  (t = {idx * 2.5:.1f} h)")
    ax.axis("off")
plt.tight_layout()
plt.show()

## 3. Model weights and label map from Mendeley (version 3, inside Colab only)

The published weights live in `GermPredModels.zip` (**4.77 GB**) and the label maps in
`GermPredRecords.zip` (911 MB) at https://doi.org/10.17632/4wkt6thgp6.3. Downloading the full archives
is unnecessary for a single-model demo, so the cell below implements a small **HTTP-Range zip reader**:
it reads the zip central directory remotely, verifies the archive size against the Mendeley metadata
(`GermPredModels.zip` sha256 `a1f1143a…`, 4,773,262,484 bytes), and streams **only** the ZeaMays frozen
inference graph member. If ranged access fails for any reason, it falls back to an explicit full
download of the archive (disclosed, ~4.8 GB). All of this runs in the Colab VM.

Mendeleyの重いzip全体を避けるため、HTTP Rangeで中央ディレクトリのみ読み取り、必要なメンバーだけを
抽出します。失敗時は明示的に全体ダウンロードへフォールバックします。

> **VERIFIED ACCESS BLOCKER (2026-10-05):** from this Colab VM, every Mendeley route returns HTTP 403 with a Cloudflare "Just a moment..." JavaScript challenge (data.mendeley.com landing, public-api, public-files download URLs, api.data.mendeley.com, and the S3 cache bucket), while GitHub raw downloads work. The challenge cannot and must not be automated, so **Run all stops at the download cell until a permitted mirror or copy is available**. No authorized public mirror was found (Zenodo/HuggingFace/Kaggle/GitHub/Wayback checked 2026-10-05). The remaining cells are kept as the complete reference implementation for the moment access is restored.
>
> **検証済みのアクセス遮断:** Colab VMからMendeleyへは全経路でCloudflare 403となり実行できません。


In [ ]:
import io, hashlib, zipfile
from pathlib import Path
import requests

MODEL_ZIP_URL = ("https://data.mendeley.com/public-files/datasets/4wkt6thgp6/files/"
                 "85cd934a-cbc7-4a01-ac09-f5e4591a05e5/file_downloaded")
MODEL_ZIP_SHA256 = "a1f1143a554062f9c17dccbeabae9effa2e527e5231227dd9c9662cd7f3acd80"
MODEL_ZIP_SIZE = 4773262484
RECORDS_ZIP_URL = ("https://data.mendeley.com/public-files/datasets/4wkt6thgp6/files/"
                   "d3f5948f-ccbf-42b0-a914-9bb2c41186a7/file_downloaded")
RECORDS_ZIP_SHA256 = "6e1f4909abcd5a770b9f23f54d09578000a5700c11aa0a32f012d37d94349d64"
RECORDS_ZIP_SIZE = 911471686

OUT_DIR = Path("/content/germination_demo")
OUT_DIR.mkdir(parents=True, exist_ok=True)


class HTTPRangeFile(io.RawIOBase):
    """Read-only file-like object over an HTTP resource with Range support (4 MB read-ahead)."""

    _BUFSIZE = 4 << 20

    def __init__(self, url):
        self.url = url
        r = requests.get(url, headers={"Range": "bytes=0-0"}, timeout=60)
        assert r.status_code == 206, f"no Range support (HTTP {r.status_code})"
        self.length = int(r.headers["Content-Range"].split("/")[-1])
        self.pos = 0
        self._buf = b""
        self._buf_start = None

    def _fetch(self, start, length):
        for attempt in range(3):
            try:
                r = requests.get(self.url,
                                 headers={"Range": f"bytes={start}-{start + length - 1}"},
                                 timeout=180)
                r.raise_for_status()
                assert r.status_code == 206, f"expected 206, got {r.status_code}"
                data = r.content
                assert len(data) == length, f"short read {len(data)} != {length}"
                return data
            except Exception:
                if attempt == 2:
                    raise
                time.sleep(2 * (attempt + 1))

    def seekable(self):
        return True

    def readable(self):
        return True

    def seek(self, offset, whence=0):
        if whence == 0:
            self.pos = offset
        elif whence == 1:
            self.pos += offset
        elif whence == 2:
            self.pos = self.length + offset
        return self.pos

    def tell(self):
        return self.pos

    def read(self, n=-1):
        if n is None or n < 0:
            n = self.length - self.pos
        n = min(n, self.length - self.pos)
        if n <= 0:
            return b""
        covered = (self._buf_start is not None
                   and self._buf_start <= self.pos
                   and self.pos + n <= self._buf_start + len(self._buf))
        if not covered:
            fetch = min(max(n, self._BUFSIZE), self.length - self.pos)
            self._buf = self._fetch(self.pos, fetch)
            self._buf_start = self.pos
        off = self.pos - self._buf_start
        out = self._buf[off:off + n]
        self.pos += n
        return out


def obtain_zip(url, expected_size, expected_sha256, local_path):
    """ZipFile via HTTP-Range member access, with an explicit full-download fallback."""
    try:
        remote = HTTPRangeFile(url)
        if remote.length != expected_size:
            raise AssertionError(f"unexpected remote size {remote.length} != {expected_size}")
        zf = zipfile.ZipFile(remote)
        zf.infolist()  # force central-directory read now
        print("HTTP-Range zip access OK")
        return zf
    except Exception as e:
        print("Range access failed -> explicit full-download fallback:", repr(e))
        if not local_path.exists() or local_path.stat().st_size != expected_size:
            with requests.get(url, stream=True, timeout=300) as r:
                r.raise_for_status()
                done = 0
                with open(local_path, "wb") as f:
                    for chunk in r.iter_content(chunk_size=1 << 22):
                        f.write(chunk)
                        done += len(chunk)
                        if done % (100 << 20) < (1 << 22):
                            print(f"  downloaded {done/1e9:.2f} GB ...", flush=True)
        h = hashlib.sha256()
        with open(local_path, "rb") as f:
            for chunk in iter(lambda: f.read(1 << 22), b""):
                h.update(chunk)
        assert h.hexdigest() == expected_sha256, "fallback download failed checksum"
        print("full download verified:", h.hexdigest())
        return zipfile.ZipFile(local_path)


model_zf = obtain_zip(MODEL_ZIP_URL, MODEL_ZIP_SIZE, MODEL_ZIP_SHA256,
                      OUT_DIR / "GermPredModels.zip")
candidates = [i.filename for i in model_zf.infolist()
              if "ZeaMays" in i.filename and i.filename.endswith("frozen_inference_graph.pb")]
print("frozen-graph candidates:", candidates)

**Extract the ZeaMays frozen inference graph.** The member is selected deterministically; `zipfile`
verifies the member CRC-32 during extraction, and the extracted size is printed. Expected size for an
Inception-ResNet-v2 frozen graph is on the order of 200–250 MB.

ZeaMaysの凍結推論グラフを抽出します。CRCは`zipfile`が検証します。

In [ ]:
assert len(candidates) >= 1, "no ZeaMays frozen_inference_graph.pb found in the archive"
# Deterministic pick: prefer a 'bestmodels' path if present, else the first sorted match.
preferred = sorted(c for c in candidates if "bestmodels" in c.lower()) or sorted(candidates)
member = preferred[0]
info = model_zf.getinfo(member)
print(f"selected member: {member}  ({info.file_size} bytes)")

MODEL_PATH = OUT_DIR / "frozen_inference_graph.pb"
if not MODEL_PATH.exists() or MODEL_PATH.stat().st_size != info.file_size:
    MODEL_PATH.write_bytes(model_zf.read(member))  # CRC-32 verified by zipfile on read
print("extracted:", MODEL_PATH, MODEL_PATH.stat().st_size, "bytes")

**Label map** `zm.pbtxt` from `GermPredRecords.zip` (same Range/fallback mechanism). Its verbatim
content is printed so the class ids are grounded in the authors' file, not invented. The author code
(`gen_records.py` / `germ_pred_util.py`) maps labels containing `_im` to class 1 and `_el` to class 2.

ラベルマップzm.pbtxtをGermPredRecords.zipから抽出し、内容をそのまま表示します。

In [ ]:
records_zf = obtain_zip(RECORDS_ZIP_URL, RECORDS_ZIP_SIZE, RECORDS_ZIP_SHA256,
                        OUT_DIR / "GermPredRecords.zip")
pbtxt_candidates = [i.filename for i in records_zf.infolist()
                    if "ZeaMays" in i.filename and i.filename.endswith(".pbtxt")]
print("pbtxt candidates:", pbtxt_candidates)
assert pbtxt_candidates, "no ZeaMays .pbtxt label map found"
pbtxt_member = ([c for c in pbtxt_candidates if c.endswith("zm.pbtxt")] or pbtxt_candidates)[0]
PBTXT_TEXT = records_zf.read(pbtxt_member).decode("utf-8")
print(f"--- {pbtxt_member} ---")
print(PBTXT_TEXT)

**Parse the label map** (plain regex — the TF-OD `label_map_util` is not pip-installable for TF 2.x, so
this small replacement is used; behavior: id → display name, matching `convert_label_map_to_categories`).

ラベルマップを正規表現で解析します（TF2ではOD APIが導入できないための最小代替です）。

In [ ]:
import re

LABEL_MAP = {}
for block in re.findall(r"item\s*\{(.*?)\}", PBTXT_TEXT, re.S):
    name = re.search(r'name:\s*"([^"]+)"', block)
    idm = re.search(r"id:\s*(\d+)", block)
    LABEL_MAP[int(idm.group(1))] = name.group(1)
print("label map:", LABEL_MAP)
assert set(LABEL_MAP) == {1, 2}, "expected exactly two classes (paper: germinated / non-germinated)"

## 4. Load the frozen detection graph (author code, cited)

The graph is loaded exactly as in the authors' `predict_image.py`
(`tf.compat.v1.Graph` + `GraphDef.ParseFromString` + `tf.import_graph_def`); a TF1 frozen GraphDef
runs fine inside TensorFlow 2.x via `tf.compat.v1`. The inference helper below is the authors'
`run_inference_for_single_image` (`scripts/gp_utils/germ_pred_util.py`, MIT) copied verbatim minus the
unused instance-mask branch (Faster R-CNN boxes only).

凍結グラフを著者コードと同じ方法で読み込み、推論ヘルパーは著者の関数を引用します。

In [ ]:
detection_graph = tf.compat.v1.Graph()
with detection_graph.as_default():
    od_graph_def = tf.compat.v1.GraphDef()
    with tf.io.gfile.GFile(str(MODEL_PATH), "rb") as fid:
        od_graph_def.ParseFromString(fid.read())
        tf.import_graph_def(od_graph_def, name="")
print("graph loaded:", len(detection_graph.get_operations()), "ops")


def run_inference_for_single_image(image, graph):
    """Author helper (grimmlab/GerminationPrediction@313305ae, germ_pred_util.py, MIT),
    verbatim minus the unused detection-masks branch."""
    with graph.as_default():
        with tf.compat.v1.Session() as sess:
            ops = tf.compat.v1.get_default_graph().get_operations()
            all_tensor_names = {output.name for op in ops for output in op.outputs}
            tensor_dict = {}
            for key in ["num_detections", "detection_boxes",
                        "detection_scores", "detection_classes"]:
                tensor_name = key + ":0"
                if tensor_name in all_tensor_names:
                    tensor_dict[key] = tf.compat.v1.get_default_graph().get_tensor_by_name(tensor_name)
            image_tensor = tf.compat.v1.get_default_graph().get_tensor_by_name("image_tensor:0")
            output_dict = sess.run(tensor_dict, feed_dict={image_tensor: np.expand_dims(image, 0)})
            output_dict["num_detections"] = int(output_dict["num_detections"][0])
            output_dict["detection_classes"] = output_dict["detection_classes"][0].astype(np.uint8)
            output_dict["detection_boxes"] = output_dict["detection_boxes"][0]
            output_dict["detection_scores"] = output_dict["detection_scores"][0]
    return output_dict

**Run inference on all 31 frames.** The score threshold 0.5 matches the author's
`visualize_boxes_and_labels_on_image_array` default used in `predict_image.py`. Per-frame wall time is
printed (GPU vs CPU differences are expected).

31フレーム全てに対して推論を実行します。スコア閾値0.5は著者コードの既定値と同じです。

In [ ]:
MIN_SCORE = 0.5

raw_outputs = []
t_start = time.time()
for i, path in enumerate(frame_paths):
    image_np = np.array(Image.open(path))
    out = run_inference_for_single_image(image_np, detection_graph)
    raw_outputs.append((image_np, out))
    if (i + 1) % 5 == 0:
        print(f"frame {i + 1}/{len(frame_paths)} done (elapsed {time.time() - t_start:.1f} s)")

elapsed = time.time() - t_start
print(f"total inference time: {elapsed:.1f} s on "
      f"{'GPU' if tf.config.list_physical_devices('GPU') else 'CPU'} "
      f"({elapsed / len(frame_paths):.2f} s/frame)")

detections = []
for image_np, out in raw_outputs:
    keep = out["detection_scores"] > MIN_SCORE
    detections.append({
        "boxes": out["detection_boxes"][keep],
        "classes": out["detection_classes"][keep],
        "scores": out["detection_scores"][keep],
    })
counts = [(d["classes"] == c).sum() for d in detections for c in (1, 2)]
counts = np.array(counts, dtype=int).reshape(len(frame_paths), 2)
print("per-frame [class1, class2] counts:\n", counts.T)

**Detection overlays (added visualization).** The authors' README color code for *predictions* is
**lime = non-germinated**, **pink = germinated**. Their script draws overlays via the TF-OD
`visualization_utils`; since that package is unavailable for TF 2.x, an explicit matplotlib box drawer
is used here instead — an adaptation that keeps the same color semantics. Which class id means
"germinated" is **inferred from the data** (the germinated count must increase over the time series)
and cross-checked with the printed label-map names above.

検出結果の重ね描き。色は著者READMEの規約（lime=非発芽、pink=発芽）に従います。どちらのクラスidが
発芽かは時系列でのカウント増加から判定します。

In [ ]:
germinated_class = int(np.argmax(counts[-1] - counts[0]) + 1)
non_class = 2 if germinated_class == 1 else 1
print(f"data-driven mapping: class {germinated_class} = germinated, class {non_class} = non-germinated")
COLORS = {germinated_class: "pink", non_class: "lime"}


def draw_overlay(ax, image_np, det):
    ax.imshow(image_np)
    h, w = image_np.shape[:2]
    for box, cls, score in zip(det["boxes"], det["classes"], det["scores"]):
        ymin, xmin, ymax, xmax = box
        color = COLORS[int(cls)]
        ax.add_patch(plt.Rectangle((xmin * w, ymin * h), (xmax - xmin) * w, (ymax - ymin) * h,
                                   fill=False, edgecolor=color, linewidth=1.5))
        ax.text(xmin * w, max(10, ymin * h - 6), f"{LABEL_MAP[int(cls)]} {score:.2f}",
                color=color, fontsize=6,
                bbox=dict(facecolor="black", alpha=0.5, pad=0.5))


fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, idx in zip(axes, [0, 10, 19]):
    draw_overlay(ax, raw_outputs[idx][0], detections[idx])
    ax.set_title(f"predictions, frame {idx * 5 + 1} (t = {idx * 2.5:.1f} h)")
    ax.axis("off")
fig.suptitle("Model predictions (added visualization) — pink = germinated, lime = non-germinated")
plt.tight_layout()
plt.show()

## 5. Germination curve from detections (added analysis — not author code)

The repository does not ship the germination-index computation, so this section is an **added,
standard-definition analysis built on the model's outputs**: seeds are linked across frames by greedy
IoU matching; a seed's germination time is the first frame at which it is predicted *germinated*
(2.5 h resolution). From those event times we report the standard **MGT** (mean germination time) and
**t50** (time to 50 % germination). These are notebook-added values, not author-published results.

ここからの発芽曲線・MGT・t50は本ノートブックの追加解析であり、著者コードではありません。

In [ ]:
def iou(a, b):
    ay1, ax1, ay2, ax2 = a
    by1, bx1, by2, bx2 = b
    iy1, ix1 = max(ay1, by1), max(ax1, bx1)
    iy2, ix2 = min(ay2, by2), min(ax2, bx2)
    inter = max(0.0, iy2 - iy1) * max(0.0, ix2 - ix1)
    union = ((ay2 - ay1) * (ax2 - ax1) + (by2 - by1) * (bx2 - bx1) - inter)
    return inter / union if union > 0 else 0.0


# Greedy frame-to-frame IoU tracking of detected seeds.
tracks = [{"box": b, "hist": [int(c)], "last": 0}
          for b, c in zip(detections[0]["boxes"], detections[0]["classes"])]
for fi in range(1, len(frame_paths)):
    det = detections[fi]
    used = set()
    for tr in tracks:
        if fi - tr["last"] > 3:
            continue
        best, bi = 0.0, None
        for j, b in enumerate(det["boxes"]):
            if j in used:
                continue
            v = iou(tr["box"], b)
            if v > best:
                best, bi = v, j
        if bi is not None and best > 0.2:
            used.add(bi)
            tr["box"] = det["boxes"][bi]
            tr["hist"].append(int(det["classes"][bi]))
            tr["last"] = fi
print(f"tracked {len(tracks)} seeds from frame 1")

FRAME_H = 2.5  # shipped frames are every 5th capture of the 30-min series
times_h = np.arange(len(frame_paths)) * FRAME_H
germ_times = []
for tr in tracks:
    hit = [k for k, c in enumerate(tr["hist"]) if c == germinated_class]
    if hit:
        germ_times.append(hit[0] * FRAME_H)
MGT = float(np.mean(germ_times)) if germ_times else float("nan")
germ_counts = counts[:, germinated_class - 1]
total_seeds = counts[:, non_class - 1][0]
t50 = float(np.interp(0.5, germ_counts / max(total_seeds, 1), times_h))
print(f"seeds tracked: {len(tracks)} | germinated by end: {len(germ_times)}")
print(f"MGT = {MGT:.2f} h | t50 = {t50:.2f} h  (added analysis, 2.5 h resolution)")

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(times_h, germ_counts, "o-", color="deeppink", label="predicted germinated seeds")
ax.plot(times_h, total_seeds - germ_counts, "s--", color="limegreen",
        label="predicted non-germinated seeds")
ax.axvline(MGT, color="gray", ls=":", label=f"MGT = {MGT:.2f} h")
ax.axvline(t50, color="black", ls=":", label=f"t50 = {t50:.2f} h")
ax.set_xlabel("time (h) from first frame")
ax.set_ylabel("number of seeds")
ax.set_title("zm1_11 germination curve from frozen-graph detections (added analysis)")
ax.legend()
plt.tight_layout()
plt.show()

**Results table and CSV export** (saved inside the Colab VM; values are model predictions on this
single petri dish, not paper-verified results).

結果テーブルとCSV出力（Colab VM内に保存）。

In [ ]:
import pandas as pd

results = pd.DataFrame({
    "frame": np.arange(1, len(frame_paths) + 1),
    "time_h": times_h,
    "predicted_germinated": germ_counts,
    "predicted_non_germinated": counts[:, non_class - 1],
})
csv_path = OUT_DIR / "zm1_11_germination_predictions.csv"
results.to_csv(csv_path, index=False)
print("saved:", csv_path)
results

## 6. Interpretation and limitations

- The released ZeaMays frozen graph detects seeds and their germination state on a previously unseen
  (to this notebook) petri-dish series without any color-threshold tuning, matching the paper's
  qualitative claim; prediction overlays and the germination curve are computed live.
- **Not verified here:** dataset-level mAP (paper: 97.90 % for Inception-ResNet v2 on maize), training,
  comparison against manual 6/12/24-h counts, and the other species' models. MGT/t50 use standard
  definitions on predicted events at 2.5-h resolution and are **added analyses**, not author code.
- Single example (`zm1_11`) only; per the paper, small bounding boxes and ambiguous transition-phase
  seeds remain the main error sources.

**References:** paper DOI 10.1186/s13007-020-00699-x (PMC7754596, CC BY 4.0); code
github.com/grimmlab/GerminationPrediction @ `313305ae…` (MIT); data/weights
doi:10.17632/4wkt6thgp6.3 (Mendeley Data v3).

日本語のまとめ: 公開済みモデルを1系列に適用する部分デモであり、論文のデータセット規模評価や学習は
含みません。検出は色しきい値なしで機能し、MGT・t50は追加解析として算出しています。